
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 데모 - 정리, 변환, 청크 파싱된 텍스트

## 개요
이 데모에서는 언어 모델과 효과적으로 활용하기 위해 파싱된 문서 텍스트를 **정리**하고 **변환**하는 방법, 그리고 검색 워크플로우를 위해 텍스트를 **분할**하는 방법을 배웁니다. 파싱된 텍스트는 현재 JSON 형식이며, 이를 일반적이고 깔끔한 텍스트로 변환하는 두 가지 방법을 시연하겠습니다:

## 학습 목표
이 데모가 끝날 때쯤이면 다음을 할 수 있게 됩니다:
1. 구문 분석된 JSON 텍스트를 LLM에 적합한 깔끔한 마크다운 형식 또는 일반 텍스트로 **변환**합니다.
2. 두 가지 변환 방법을 **비교**합니다: LLM 기반 의미론적 정리와 빠른 연결.
3. LangChain을 사용하여 정제된 텍스트를 페이지 단위로 **분할**하되, 문맥을 위해 일부 중첩을 허용합니다.
4. 하위 작업 임베딩 및 AI Search를 위해 최종 분할된 테이블을 **저장**합니다.

## 요구 사항
* JSON 형식의 **파싱된 문서 테이블**. 이 표는 이전 데모에서 만들어졌습니다. 아직 완료하지 않았다면, **먼저 이 데모를 완료해야 합니다 (`2.2 Demo - Parse Documents to Structured Data`)**.
* **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.
* 필요한 라이브러리는 서버리스 Compute 구성의 **의존성**에 추가됩니다.

## 준비

아래 코드를 실행하여 교실 환경을 설정하세요. 이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.

In [0]:
%run ../Includes/Classroom-Setup-02

## A. 파싱된 JSON을 깨끗한 텍스트로 변환

이 섹션에서는 구문 분석된 JSON 텍스트를 언어 모델에서 사용할 수 있는 깨끗하고 일반 텍스트로 변환할 것입니다. 두 가지 방법을 시연하겠습니다:

1. **LLM 기반 의미 정리**를 사용하여 `ai_query` JSON을 마크다운 텍스트로 배치 처리하고 변환합니다. 이 방법은 문서 의미를 더 많이 보존하지만 비용이 더 많이 들 수 있습니다.
2. 모든 텍스트 요소를 하나의 평문 문자열로 연결하는 **빠른 연결** 기능. 이 방법은 빠르고 비용 효율적이지만, 일부 의미 구조(예: 페이지 헤더)가 손실됩니다.

*두 방법 모두 `== page ==` 토큰을 사용하여 나중에 청크하고 검색할 수 있도록 페이지를 분리합니다. Workflows.*

### A1. 파싱된 문서 로드

먼저 이전 데모에서 생성된 파싱된 문서를 불러오는 것부터 시작해 봅시다. 이 단계는 정리와 변환에 필요한 구조화된 데이터를 확보할 수 있도록 보장합니다.

*알림: 진행하기 전에 파싱된 테이블이 존재하고 최신 상태인지 꼭 확인하세요.*

In [0]:
parsed_table = f"{catalog}.{schema}.docs_parsed"
chunked_table = f"{catalog}.{schema}.docs_chunked"

parsed_df = spark.read.table(parsed_table)

print(f"Loaded parsed documents from: {parsed_table}")
parsed_df.printSchema()

### A2. ai_query를 활용한 LLM 기반 의미론적 정리

이 방법에서는 `ai_query` 함수를 사용하여 파싱된 JSON 텍스트를 일괄 처리하고 이를 깔끔한 마크다운 형식의 텍스트로 변환합니다. 이 접근법은 헤더, 테이블, 구조와 같은 문서 의미론을 보존하기 위해 대형 언어 모델(LLM)을 활용하여 출력 결과를 하위 LLM 작업에 더 유용하게 만듭니다.

- **장점:** 구조와 의미를 더 잘 유지하고, 고품질 마크다운을 생산합니다.
- **단점:** LLM 사용으로 인해 비용이 더 많이 들 수 있습니다.

**⚠️ 경고**: LLM 기반 정리는 많은 문서를 처리하는 데 더 많은 비용이 발생할 수 있습니다. 빠르고 저렴한 처리를 위해 빠른 연결 방식을 사용하세요.

**참고:** 이 `ai_query` 기능은 OpenAI, Anthropic, Databricks 등 여러 기초 모델을 지원합니다. 이 데모에서는 Anthropic의 Claude Sonnet 4.6 모델을 사용할 것입니다.

LLM에 JSON를 파싱하고 깨끗한 마크다운을 출력하도록 지시하며, `== page ==`를 페이지 간 구분자로 사용합니다.

In [0]:
from pyspark.sql.functions import expr

# Databricks 기초 모델(또는 본인의 서빙 엔드포인트 이름을 선택하세요)
ENDPOINT = "databricks-claude-sonnet-4-6"

# LLM 예시 프롬프트
prompt_prefix = '''
당신은 도움이 되는 조수입니다. 페이지, 요소, 메타데이터 포함 파싱된 문서를 나타내는 JSON 객체가 주어지면, 그 내용을 깔끔하고 읽기 쉬운 마크다운으로 변환합니다. Use "== page ==" 각 페이지를 구분합니다. 헤더, 표, 캡션과 같은 중요한 구조를 유지하세요. 출력에는 JSON이나 코드 블록을 포함하지 말고, 깨끗한 마크다운 텍스트만 포함하세요.

JSON:

'''

# 신청 ai_query하여 파싱된 JSON 텍스트를 배치 처리하기
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
          ai_query(
            '{ENDPOINT}',
            CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING))
          )
        """)
    )
)

display(transformed_df.select("path", "clean_markdown_text"))

### A3. 빠른 평문 변환

이 방법에서는 파싱된 JSON의 모든 텍스트 요소를 빠르게 하나의 평문 문자열로 연결합니다. 이 방법은 빠르고 비용 효율적이지만, 그것은 헤더, 표, 캡션 같은 문서 구조를 희생합니다.

- **장점:** 빠르고, 저렴하며, 구현이 간단합니다.
- **단점:** 중요한 구조와 의미론을 잃습니다.

**참고:** 우리는 각 페이지에서 텍스트를 추출하고 조인하는 데 Spark을 사용하며, 나중에 청크화할 수 있도록 페이지 사이에 토큰을 `== page ==` 삽입합니다.

콘텐츠 추출 논리는 `Includes/content_extractor` 파일 내에 제공됩니다.

In [0]:
from pyspark.sql import functions as F

# 먼저 VARIANT/struct/map을 JSON 문자열로 변환하세요 (VariantVal 문제를 피하기 위해)
safe_json_col = F.coalesce(
    F.to_json(F.col("parsed_content")),
    F.col("parsed_content").cast("string")
)

# UDF를 적용
plain_text_df = parsed_df.withColumn(
    "plain_text",
    extract_contents_udf()(safe_json_col)
)

display(plain_text_df.select("path", "plain_text"))


## B. 검색을 위한 청크된 텍스트

이제 깨끗하고 페이지 구분된 텍스트가 있으니, 텍스트를 추출해 Workflows를 추출하겠습니다. 청킹은 언어 모델과 AI Search 시스템이 관련 정보를 효율적으로 처리하고 검색하는 데 도움을 줍니다.

LangChain의 `RecursiveCharacterTextSplitter`를 사용해 텍스트를 토큰별로 `== page ==` 나누겠습니다. 이 유틸리티는 자동으로 청킹과 겹침을 처리하여 텍스트를 임베딩과 검색에 쉽게 준비할 수 있게 합니다.

**참고:** 오버랩은 단일 입력이 여러 청크로 분할될 때만 발생합니다. 이 예시에서는 각 페이지가 일반적으로 하나의 `chunk_size=2000` 청크를 이루므로, 일부 행은 겹치지 않을 수 있습니다. 겹치는 청크를 더 명확히 관찰하려면 청크 크기를 줄여보세요.

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd

# 청킹 매개변수 설정: chunk_size 각 청크의 최대 길이를 조절하고, chunk_overlap는 청크 간 텍스트 중첩을 허용하여 검색 품질을 향상시킵니다.
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

# 선호하는 구분자를 사용하여 텍스트 분할기를 만드십시오.
# 이 분할기는 페이지 마커나 자연스러운 경계에서 텍스트를 끊어 가능한 경우 문서 구조를 유지합니다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

# 청크된 DataFrame의 출력 스키마를 정의하세요.
# 각 행은 문서 경로와 하나의 텍스트 청크를 포함합니다.
schema = StructType([
    StructField("path", StringType(), True),
    StructField("chunk", StringType(), True),
])

def split_rows(iterator):
    """
    mapInPandas 함수: 열이 있는 PDF 입력 [경로, plain_text],
    출력 행 [경로, 청크].
    이 함수는 각 문서의 텍스트를 조각으로 나누어 DataFrame 구성을 위해 제공합니다.
    """
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    if c and c.strip():
                        out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])

# 분할기를 평문 DataFrame에 적용하세요.
# 이 단계는 각 문서를 여러 개의 청크로 변환하여 효율적인 하위 검색 및 임베딩할 수 있게 합니다.
df_chunks = (
    plain_text_df
    .select("path", "plain_text")
    .mapInPandas(split_rows, schema=schema)
)

# 청크된 DataFrame을 표시해 확인해 보세요.
display(df_chunks)

## C. 챕킹된 데이터를 Delta 테이블에 저장합니다

청크된 텍스트 데이터를 Delta 테이블에 저장하여 하위 임베딩 및 Workflows에서 검색을 위한 워크플로우에 사용하겠습니다.

**태스크:** 설정 섹션에서 **`chunked_table`** 로 정의된 테이블에 청크된 DataFrame을 쓰세요.

In [0]:
from pyspark.sql import functions as F

# 저장 전에 고유하고 점진적인 id 열을 추가하세요
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# id가 포함된 청크 데이터를 Delta 테이블에 저장해 검색과 임베딩을 하세요
df_chunks.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(chunked_table)

display(spark.read.table(chunked_table))

## 요약과 다음 단계

이 데모에서는 파싱된 문서를 불러오고, **LLM 기반 의미 정리**와 **빠른 평문 추출**을 적용한 뒤, 결과를 Workflows를 위해 **청크**로 처리했습니다. 그 후 청크된 데이터를 Delta 테이블에 저장하고, AI Search과 LLM 기반 애플리케이션과의 임베딩 및 통합을 위해 그것을 준비했습니다.

**핵심 요점:**
- LLM 기반 의미 정리 또는 빠른 연결 작업을 사용하여 텍스트를 청크화할 준비합니다.
- 텍스트를 페이지별로 청크합니다.
- AI Search 및 임베딩 파이프라인과 쉽게 통합할 수 있도록 청크 데이터를 Delta 테이블에 저장합니다.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>